In [1]:
from collections import Counter
from typing import TypeAlias

Bigram: TypeAlias = tuple[int, int]

In [13]:
s = 'aa acababa'
print(f"{len(s)}, {s}")

token_ids = list(s.encode("ascii"))
print(f"{len(token_ids)}, {token_ids}")

vocab = [chr(i) for i in range(128)]
print(f"{len(vocab)}")

merge_rules = {}

10, aa acababa
10, [97, 97, 32, 97, 99, 97, 98, 97, 98, 97]
128


In [14]:
def compute_bigram_statistics(
    token_ids: list[int], counter: Counter | None = None
) -> Counter[Bigram, int]:
    bigram_counter = Counter() if counter is None else counter
    for left, right in zip(token_ids, token_ids[1:]):
        bigram_counter[(left, right)] += 1
    return bigram_counter

In [15]:
def replace_bigram(token_ids: list[int], bigram: Bigram, bigram_id: int) -> list[int]:
    idx = 0
    new_token_ids = []
    while idx < len(token_ids):
        if token_ids[idx : idx + 2] == list(bigram):
            new_token_ids.append(bigram_id)
            idx += 2
        else:
            new_token_ids.append(token_ids[idx])
            idx += 1
    return new_token_ids

In [16]:
def get_merge_tokens(token_ids):
    bigram_frequencies = {}
    bigram_frequencies = compute_bigram_statistics(token_ids)
    
    max_val = max(bigram_frequencies.values())
    filt_bigram_frequencies = {k: v for k, v in bigram_frequencies.items() if v == max_val}
    if len(filt_bigram_frequencies) > 1:
        filt_bigram_frequencies = dict(sorted(filt_bigram_frequencies.items()))
    return list(filt_bigram_frequencies.keys())[0]

In [22]:
def merge(token_ids, vocab, merge_rules):
    # get bigram frequencies
    bigram_frequencies = compute_bigram_statistics(token_ids)

    # find the most frequent bigram lexicographically
    max_val = max(bigram_frequencies.values())
    filt_bigram_frequencies = {k: v for k, v in bigram_frequencies.items() if v == max_val}
    if len(filt_bigram_frequencies) > 1:
        filt_bigram_frequencies = dict(sorted(filt_bigram_frequencies.items()))
    merged_bigram = list(filt_bigram_frequencies.keys())[0]

    # merge the tokens and update the vocab and merge rules
    new_token = vocab[merged_bigram[0]] + vocab[merged_bigram[1]]
    merged_bigram_id = len(vocab)
    vocab = vocab + [new_token]
    merge_rules[merged_bigram] = merged_bigram_id

    new_token_ids = replace_bigram(token_ids, merged_bigram, merged_bigram_id)
    return new_token_ids, vocab, merge_rules

In [23]:
new_token_ids, new_vocab, new_merge_rules = merge(token_ids, vocab, merge_rules)

In [24]:
new_merge_rules

{(97, 98): 128}

In [25]:
new_vocab[-5:]

['|', '}', '~', '\x7f', 'ab']

In [26]:
new_token_ids

[97, 97, 32, 97, 99, 128, 128, 97]